In [1]:
# ==========================================
# 필수 패키지 설치 (Colab / Production)
# ==========================================

!pip install -qU langchain-openai langgraph langchain langchain-core langchain-community \
graphviz chromadb python-docx unstructured --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
 

In [2]:
import os
from typing import TypedDict, Optional

from google.colab import userdata
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END


# ============================================================
# 1. API KEY
# ============================================================

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# ============================================================
# 2. STATE 정의 (핵심: sender_name 추가)
# ============================================================

class EmailState(TypedDict):
      email_text:str        # 입력 이메일
      sender_name:str    # 발신자 이름 (자동 추출)
      summary:str          # 이메일 요약
      draft_reply:str      # 답장 초안
      tags:str            # 분류


# ============================================================
# 3. LLM 초기화
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)


# ============================================================
# 4. Node 1 - sender_name 추출 (핵심 추가)
# ============================================================

def extract_sender(state: EmailState):

    prompt = f"""
다음 이메일에서 발신자 이름을 추출하세요.

이메일:
{state['email_text']}

규칙:
- 이름이 없으면 "고객"이라고 출력
- 반드시 한 단어로만 출력
"""

    response = llm.invoke(prompt)

    return {"sender_name" : response.content.strip()}


# ============================================================
# 5. Node 2 - 이메일 요약
# ============================================================

def summarize_email(state: EmailState):

    prompt = f"다음 이메일을 요약하세요:\n\n{state['email_text']}"

    response = llm.invoke(prompt)

    return {"summary":response.content}


# ============================================================
# 6. Node 3 - 답장 생성 (sender_name 강제 삽입)
# ============================================================

def draft_reply(state: EmailState):

    prompt = f"""
너는 기업 이메일 답장 AI다.

반드시 아래 형식을 지켜라:

[필수 규칙]
1. 첫 문장은 반드시:
   "안녕하세요 {state["sender_name"]}님,"

2. 요약:
{state["summary"]}

3. 이메일 내용:
{[state["email_text"]]}

4. 정중하고 짧게 답장 작성
"""

    response = llm.invoke(prompt)

    return {"draft_reply":response.content}


# ============================================================
# 7. Node 4 - 태깅
# ============================================================

def tag_email(state: EmailState):

    prompt = f"""
이메일을 다음 중 하나로 분류:
- 긴급
- 일반
- 홍보
- 기타

내용:
{state["email_text"]}
"""

    response = llm.invoke(prompt)

    return {"tags":response.content}


# ============================================================
# 8. LangGraph 구성
# ============================================================

workflow = StateGraph(EmailState)

workflow.add_node("extract_sender", extract_sender)
workflow.add_node("summarize_email", summarize_email)
workflow.add_node("draft_reply", draft_reply)
workflow.add_node("tag_email", tag_email)


# 흐름 정의
workflow.add_edge(START, "extract_sender")
workflow.add_edge("extract_sender", "summarize_email")
workflow.add_edge("summarize_email", "draft_reply")
workflow.add_edge("draft_reply", "tag_email")
workflow.add_edge("tag_email", END)


# 컴파일
app = workflow.compile()


# ============================================================
# 9. 실행 테스트
# ============================================================

test_email = """
김영희입니다. 이번 프로젝트 일정 확인 부탁드립니다.
빠른 회신 부탁드립니다.
"""

result = app.invoke({
    "email_text" : test_email
})


# ============================================================
# 10. 결과 출력
# ============================================================

print("\n===== RESULT =====")
print("Sender:", result["sender_name"])
print("Summary:", result["summary"])
print("Reply:\n", result["draft_reply"])
print("Tags:", result["tags"])


===== RESULT =====
Sender: 김영희
Summary: 김영희가 프로젝트 일정 확인을 요청하며 빠른 회신을 부탁하는 이메일입니다.
Reply:
 안녕하세요 김영희님,

프로젝트 일정에 대한 확인 요청 잘 받았습니다. 현재 일정을 검토 중이며, 최대한 빠른 시일 내에 회신드리겠습니다. 

감사합니다.
Tags: 이 이메일은 "긴급"으로 분류할 수 있습니다. 빠른 회신을 요청하고 있기 때문입니다.


#문제) 아래의 조건을 참고하여 { }, ( ), [ ] 안에 들어갈 코드를 완성하세요.




In [ ]:
"""

extract_sender_name:
- 역할: 이메일에서 발신자 이름 추출
- 입력: state["email_text"]
- 출력: {"sender_name": str}

summarize_email:
- 역할: 이메일 핵심 내용 요약
- 입력: state["email_text"]
- 출력: {"summary": str}

draft_reply:
- 역할: sender_name + summary 기반 답장 생성
- 입력: state["email_text"], state["summary"], state["sender_name"]
- 출력: {"draft_reply": str}

tag_email:
- 역할: 이메일 분류 (긴급/일반/홍보/기타)
- 입력: state["email_text"]
- 출력: {"tags": str}


"""

'\n\nextract_sender_name:\n- 역할: 이메일에서 발신자 이름 추출\n- 입력: state["email_text"]\n- 출력: {"sender_name": str}\n\nsummarize_email:\n- 역할: 이메일 핵심 내용 요약\n- 입력: state["email_text"]\n- 출력: {"summary": str}\n\ndraft_reply:\n- 역할: sender_name + summary 기반 답장 생성\n- 입력: state["email_text"], state["summary"], state["sender_name"]\n- 출력: {"draft_reply": str}\n\ntag_email:\n- 역할: 이메일 분류 (긴급/일반/홍보/기타)\n- 입력: state["email_text"]\n- 출력: {"tags": str}\n\n\n'

In [3]:
import os  # 환경변수 설정
import gradio as gr  # UI 구성
from typing import TypedDict  # LangGraph State 정의

from google.colab import userdata  # Colab secret 관리
from langchain_openai import ChatOpenAI  # LLM 모델
from langgraph.graph import StateGraph, START, END  # LangGraph 구조

# ============================================================
# 1. API KEY 설정
# ============================================================

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # OpenAI 키 로드

# ============================================================
# 2. State 정의 (LangGraph 공유 메모리)
# ============================================================

class EmailState(TypedDict):  # 전체 workflow 상태 정의
    email_text: str          # 입력 이메일
    sender_name: str         # 발신자 이름
    summary: str             # 이메일 요약
    draft_reply: str         # 답장
    tags: str                # 이메일 분류

# ============================================================
# 3. LLM 초기화
# ============================================================

llm = ChatOpenAI(  # GPT 모델 초기화
    model="gpt-4o-mini",  # 경량 모델
    temperature=0.3       # 안정적인 출력
)

# ============================================================
# 4. Node 1: sender 추출
# ============================================================

def extract_sender_name(state: EmailState):  # 발신자 이름 추출 Node

    prompt = f"""
이메일에서 발신자 이름만 추출하세요.

규칙:
- 이름만 출력
- 없으면 "고객"

이메일:
{ state["email_text"]}
"""  # 이름 추출 프롬프트

    response = llm.invoke(prompt)  # LLM 실행

    return { "sender_name" : response.content.strip() }  # state 업데이트

# ============================================================
# 5. Node 2: 이메일 요약
# ============================================================

def summarize_email(state: EmailState):  # 이메일 요약 Node

    prompt = f"""
다음 이메일을 핵심만 요약하세요:

{ state["email_text"]}
"""  # 요약 프롬프트

    response = llm.invoke(prompt)  # LLM 호출

    return { "summary": response.content.strip() }  # state 업데이트

# ============================================================
# 6. Node 3: 답장 생성 (핵심)
# ============================================================

def draft_reply(state: EmailState):  # 답장 생성 Node

    prompt = f"""
너는 기업 이메일 AI다.

반드시 첫 문장은:
"안녕하세요 { state["sender_name"]}님,"

이메일:
{ state["email_text"]}

요약:
{ state["summary"]}

정중하고 간결하게 답변 작성.
"""  # 답장 생성 프롬프트

    response = llm.invoke(prompt)  # LLM 호출

    return {  "draft_reply" : response.content}  # state 업데이트

# ============================================================
# 7. Node 4: 이메일 태깅
# ============================================================

def tag_email(state: EmailState):  # 이메일 분류 Node

    prompt = f"""
이 이메일을 다음 중 하나로 분류하세요:
- 긴급
- 일반
- 홍보
- 기타

이메일:
{ state["email_text"]}
"""  # 분류 프롬프트

    response = llm.invoke(prompt)  # LLM 실행

    return { "tags": response.content.strip()}  # state 업데이트

# ============================================================
# 8. LangGraph Workflow 구성
# ============================================================

workflow = StateGraph(EmailState)  # State 기반 그래프 생성

workflow.add_node("extract_sender_name", extract_sender_name)  # sender Node
workflow.add_node("summarize_email", summarize_email)          # summary Node
workflow.add_node("draft_reply", draft_reply)                  # reply Node
workflow.add_node("tag_email", tag_email)                      # tag Node

workflow.add_edge(START,  "extract_sender_name")  # 시작 → sender
workflow.add_edge( "extract_sender_name",  "summarize_email")  # sender → summary
workflow.add_edge("summarize_email",  "draft_reply")  # summary → reply
workflow.add_edge("draft_reply",  "tag_email")  # reply → tag
workflow.add_edge("tag_email"  , END)  # 종료

app = workflow.compile()  # 실행 가능한 그래프 생성

# ============================================================
# 9. 실행 함수 (Gradio용)
# ============================================================

def run_agent(email_text):  # UI 입력 처리 함수

    result = app.invoke({  # LangGraph 실행
        "email_text": email_text  # 초기 input
    })

    return (
        result[ "sender_name"],  # 발신자
        result[ "summary"],      # 요약
        result[ "draft_reply"],  # 답장
        result[ "tags"]         # 태그
    )

# ============================================================
# 10. Gradio UI
# ============================================================

with gr.Blocks() as demo:  # UI container

    gr.Markdown("## LangGraph Email Agent (Production V2)")  # 제목

    email_input = gr.Textbox(lines=6, label="이메일 입력")  # 입력창

    sender_out = gr.Textbox(label="발신자")  # 출력 1
    summary_out = gr.Textbox(label="요약")   # 출력 2
    reply_out = gr.Textbox(label="답장")     # 출력 3
    tags_out = gr.Textbox(label="태그")      # 출력 4

    btn = gr.Button("실행")  # 실행 버튼

    btn.click(  # 버튼 이벤트
        fn=run_agent,  # 실행 함수
        inputs=email_input,  # 입력
        outputs=[sender_out, summary_out, reply_out, tags_out]  # 출력
    )

demo.launch()  # 앱 실행

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://47d3dca0fb2de95758.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
"""

Email Generator

Email Generator Agent:
- 역할: 사용자 이메일을 받아 LLM이 정중한 비즈니스 이메일로 재작성
- 입력: user_email (string)
- 출력: formatted_email (string)

load_language_model:
- 역할: ChatOpenAI 모델 초기화
- 출력: LLM 인스턴스

generate_email:
- 역할: Prompt + LLM 실행하여 이메일 생성

"""

In [5]:
import os  # 환경변수 관리

from google.colab import userdata  # Colab secret 접근
from langchain_openai import ChatOpenAI  # 최신 LLM
from langchain_core.prompts import PromptTemplate  # Prompt 템플릿

# ============================================================
# 1. API KEY 설정
# ============================================================

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # API KEY 로드

# ============================================================
# 2. 사용자 입력
# ============================================================

user_email = input("이메일 내용을 입력하세요: ")  # 사용자 입력 받기

# ============================================================
# 3. Prompt Template 정의
# ============================================================

query_template = """
당신은 전문 비즈니스 이메일 작성 AI입니다.

아래 이메일을 정중하고 자연스럽게 개선하세요.

이메일:
{email}
"""

prompt = PromptTemplate(  # Prompt 객체 생성
     input_variables= ["email"],   # 입력 변수 정의
     template = query_template   # 템플릿 설정
)

# ============================================================
# 4. LLM 로드 함수 (Production 스타일)
# ============================================================

def load_language_model():  # LLM 초기화 함수
    return ChatOpenAI(
        model = "gpt-4o-mini",
        temperature=0
    )


# ============================================================
# 5. 이메일 생성 함수
# ============================================================

def generate_email(user_email: str):  # 이메일 생성 Agent
    llm = load_language_model()
    formatted_prompt = prompt.format(email= user_email)
    response = llm.invoke(formatted_prompt)
    return response.content  # 결과 반환

# ============================================================
# 6. 실행 로직
# ============================================================

if user_email:  # 입력이 존재할 경우 실행

    result = generate_email(user_email)  # 이메일 생성

    print("\n생성된 이메일")  # 결과 출력
    print("-" * 50)
    print(result)

else:  # 입력이 없는 경우

    print("이메일 내용을 입력해주세요.")  # 에러 메시지


이메일 내용을 입력하세요: 산행을 같이 할 신입 동호회 회원을 모집합니다.

생성된 이메일
--------------------------------------------------
제목: 신입 동호회 회원 모집 안내

안녕하세요,

저희 동호회에서는 함께 산행을 즐길 신입 회원을 모집하고 있습니다. 자연 속에서의 소중한 경험을 나누고, 새로운 친구들을 만날 수 있는 기회를 제공하고자 합니다.

관심이 있으신 분들은 언제든지 연락 주시기 바랍니다. 많은 참여 부탁드립니다!

감사합니다.

[당신의 이름]  
[동호회 이름]  
[연락처]  


In [ ]:
"""

Email Writing Agent:

- 역할:
  사용자의 이메일 초안을 받아 정중하고 전문적인 이메일로 변환

- 구성:
  1. PromptTemplate → 이메일 스타일 정의
  2. ChatOpenAI → LLM 생성
  3. ConversationBufferMemory → 대화 기록 유지
  4. LangChain Runnable → 실행 체인

- 입력:
  draft (str)

- 출력:
  formatted_email (str)


"""

In [6]:
import os  # 환경변수 관리
import gradio as gr  # UI
from google.colab import userdata  # Colab secret

from langchain_openai import ChatOpenAI  # 최신 LLM
from langchain_core.prompts import PromptTemplate  # Prompt
from langchain_core.runnables import RunnableLambda  # LCEL 실행용

# ============================================================
# 1. API KEY 설정
# ============================================================

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # API KEY 로드

# ============================================================
# 2. Prompt Template
# ============================================================

email_prompt = PromptTemplate(
    input_variables=["draft"],
    template=(
        "당신은 전문 이메일 작성 AI입니다.\n"
        "아래 초안을 정중하고 자연스럽게 수정하세요.\n\n"
        "초안:\n{draft}\n\n"
        "최종 이메일:"
    )
)

# ============================================================
# 3. LLM 초기화 (최신 방식)
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# ============================================================
# 4. LCEL 체인 구성 (핵심 변경)
# ============================================================

chain = email_prompt | llm

# ============================================================
# 5. Agent 함수
# ============================================================

def generate_email_agent(user_input: str):

    # 입력 검증
    if not user_input.strip():
        return "이메일 초안을 입력해주세요."

    # LCEL 실행 (invoke)
    response = chain.invoke({"draft" : user_input})

    return response.content  # LLM 출력

# ============================================================
# 6. Gradio UI
# ============================================================

with gr.Blocks() as demo:

    gr.Markdown("## GPT 이메일 작성 에이전트 (LCEL 최신 버전)")

    email_input = gr.Textbox(
        label="초안 입력",
        placeholder="예: 회의 일정 다시 확인 부탁드립니다.",
        lines=5
    )

    generate_btn = gr.Button("이메일 생성")

    email_output = gr.Textbox(
        label="작성된 이메일",
        lines=10
    )

    generate_btn.click(
        fn=generate_email_agent,
        inputs=email_input,
        outputs=email_output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a3d9eb7b04809177fc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
"""

메모리 기반

TEST_HEADER =

From: 유관순 <ryu@example.com>
To: 홍길동 <hong@example.com>
Date: 2025-10-24 09:15:00 +0900
Subject: 회의 일정 확인 요청


TEST_BODY =

안녕하세요 홍길동 교수님,

2025년 교직원을 위한 아동학대 예방교육의 일정이 촉박하여 다시 한번 안내드리니 정해진 기간 내에 이수하여주시기 바랍니다.

※ 타 기관에서 이미 이수하신 경우, 이수증을 메일로 회신주시기 바랍니다



1. 교육 개요

  가. 과정명: 2025년 교직원 대상 아동학대 예방교육

  나. 기간: 2025.05.01.(목) ~ 2025.12.31.(수)

  ※ 법정 의무 수강 기간이 2025.12.31.까지로 기간 연장이 불가함을 유의

  다. 대상: 대학 교직원

  라. 방식: 온라인 동영상 강의(한국어, 영어, 제공)

  마. 내용: 아동학대의 정의 및 관련 법, 학대유형 및 의심징후, 신고 및 대응체계 등

  바. 이수기준 및 이수증: 언어별 1개 강의 100% 이수 시 이수증 발급



2. 교육 이수 방법(붙임의 매뉴얼 참조)

  로그인 → LMS 바로가기 (왼쪽 메뉴 영역 중간) → 마이페이지 (Learning)

  → 2025년 교직원 대상 아동학대 예방교육→ 과목 홈 바로가기 → 주차학습

  → 강의 수강 (한국어, 영어 중 택1 가능)



3. 유의사항

  가. 전임교원 이수결과 교육업적평가 반영

  나. 정규직 직원 이수결과 직원인사고과 평가 반영

  다. 학습기간 중 미이수자에 대한 독려 문자 및 메일 발송 예정.



학부대학 교육운영팀
유관순 조교/ 02-000-1111

"""

In [ ]:
# ==========================================
# 라이브러리 임포트
# ==========================================
import os  # 환경변수 설정용
import gradio as gr  # UI 프레임워크

from google.colab import userdata  # Colab secret 접근

from langchain_openai import ChatOpenAI  # 최신 OpenAI LLM
from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿
from langchain_core.output_parsers import StrOutputParser  # 문자열 출력 변환

# ==========================================
# API 키 설정
# ==========================================
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")  # Colab secret에서 키 로드

# ==========================================
# LLM 초기화
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)  # 안정적인 결정형 출력

# ==========================================
# LCEL 실행 헬퍼 함수
# ==========================================
def run_chain(template_str, input_vars):
    """
    [역할]
    Prompt → LLM → String 변환까지 한 번에 처리하는 공통 실행기

    [구조]
    PromptTemplate → ChatOpenAI → StrOutputParser
    """

    # PromptTemplate 생성 (입력 변수 자동 매핑)
    prompt = PromptTemplate(
        input_variables=
        template=
    )

    # LCEL 체인 구성 (LangChain 최신 방식)
    chain =

    # 실행 (invoke)
    return


# ==========================================
# 1. 이메일 헤더 파싱 Agent
# ==========================================
def parse_email_header(raw_header):
    """
    [Agent 역할]
    - 이메일 헤더에서 발신자 이름 + 수신일 추출
    - 정보 구조화 단계 (Information Extraction Agent)
    """

    template = """헤더에서 발신자 이름과 수신일시를 추출하세요.
출력형식:
Sender Name: <이름>
Received Date: <날짜>

헤더:
{raw_header}"""

    out =

    sender, date =


    # 결과 파싱
    for line in out.splitlines():
        if "Sender Name:"  in line:
              sender = line.split(":", 1)[1].strip()
        if "Recieved Date:" in line:
              date = line.split(":", 1)[1].strit()
    return sender, date


# ==========================================
# 2. 수신자 역할 추론 Agent
# ==========================================
def detect_recipient_role(raw_header, email_text):
    """
    [Agent 역할]
    - 이메일 내용 + 헤더 기반으로 수신자 관계 추론
    - Classification Agent
    """

    template = """이메일 헤더와 본문을 참고하여 수신자 역할을 추론하세요.
범주: 상사, 동료, 고객, 공급업체, 기타

출력 형식:
Role: <역할>

헤더:
{raw_header}

본문:
{email_text}"""

    out =


    # Role 값 추출
    return


# ==========================================
# 3. 이메일 요약 Agent
# ==========================================
def summarize_email(email_text):
    """
    [Agent 역할]
    - 이메일 핵심 내용 압축
    - Summarization Agent
    """

    template = "다음 이메일을 간결하게 요약하세요:\n\n{email_text}"

    return run_chain(template, {"email_text": email_text})


# ==========================================
# 4. 답변 생성 Agent
# ==========================================
def draft_reply(email_text, summary, sender, date, role):
    """
    [Agent 역할]
    - 구조화된 정보를 기반으로 답변 이메일 생성
    - Generation Agent (핵심 LLM Agent)
    """

    template = """이메일 내용: {email_text}
요약: {summary}
발신자: {sender}
수신일: {date}
수신자 역할: {role}

위 정보를 기반으로 전문적이고 간결한 답변 초안을 작성하세요."""

    return run_chain(
        template,
        {


        }
    )


# ==========================================
# 5. 이메일 태그 Agent
# ==========================================
def tag_email(email_text, summary):
    """
    [Agent 역할]
    - 이메일 유형 분류
    - Classification Agent
    """

    template = """이메일 내용과 요약을 기반으로
'긴급', '일반', '홍보', '기타' 중 하나로 태깅하세요.

출력 형식:
Tag: <태그>

내용:
{email_text}

요약:
{summary}"""

    out = run_chain(

    )

    return out.split("Tag: ")[-1].strip() if "Tag: " in out else "기타"


# ==========================================
# 전체 Agent Workflow (Orchestrator)
# ==========================================
def process_email(raw_header, email_text):
    """
    [역할]
    - 모든 Agent를 순차 실행하는 Orchestrator
    - Pipeline Controller 역할
    """

    if not email_text.strip():
        return ("", "", "", "", "내용을 입력하세요.", "")

    # Step 1: 헤더 분석
    sender, date =

    # Step 2: 역할 추론
    role =

    # Step 3: 요약
    summary =

    # Step 4: 답변 생성
    reply =

    # Step 5: 태그 분류
    tag =

    return sender, date, role, summary, reply, tag


# ==========================================
# Gradio UI
# ==========================================
with gr.Blocks() as demo:

    gr.Markdown("## 📧 최신 LangChain 0.3+ 이메일 Agent")

    # 입력 UI
    h_input = gr.Textbox(label="이메일 헤더", lines=3)
    b_input = gr.Textbox(label="이메일 본문", lines=8)

    btn = gr.Button("실행")

    # 출력 UI
    with gr.Row():
        out1 = gr.Textbox(label="발신자")
        out2 = gr.Textbox(label="수신일")
        out3 = gr.Textbox(label="역할")

    out4 = gr.Textbox(label="요약")
    out5 = gr.Textbox(label="답변 초안", lines=5)
    out6 = gr.Textbox(label="태그")

    # 실행 연결
    btn.click(
        process_email,
        inputs=[h_input, b_input],
        outputs=[out1, out2, out3, out4, out5, out6]
    )

# 실행
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://edfdfcbf08f3b0218e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#문제) LCEL 기반의 메일분석기입니다. 다음 비어있는 빈칸을 완성하세요.

In [ ]:
"""
LCEL-based Sequential Multi-Agent Pipeline

Input Email
   ↓
Split Agent
   ↓
Header Parsing Agent
   ↓
Role Detection Agent
   ↓
Summarization Agent
   ↓
Reply Generation Agent
   ↓
Tagging Agent
   ↓
Output


메일 원문 :

From: 유관순 <ryu@example.com>
To: 홍길동 <hong@example.com>
Date: 2025-10-24 09:15:00 +0900
Subject: 회의 일정 확인 요청


안녕하세요 홍길동 교수님,

2025년 교직원을 위한 아동학대 예방교육의 일정이 촉박하여 다시 한번 안내드리니 정해진 기간 내에 이수하여주시기 바랍니다.

※ 타 기관에서 이미 이수하신 경우, 이수증을 메일로 회신주시기 바랍니다



1. 교육 개요

  가. 과정명: 2025년 교직원 대상 아동학대 예방교육

  나. 기간: 2025.05.01.(목) ~ 2025.12.31.(수)

  ※ 법정 의무 수강 기간이 2025.12.31.까지로 기간 연장이 불가함을 유의

  다. 대상: 대학 교직원

  라. 방식: 온라인 동영상 강의(한국어, 영어, 제공)

  마. 내용: 아동학대의 정의 및 관련 법, 학대유형 및 의심징후, 신고 및 대응체계 등

  바. 이수기준 및 이수증: 언어별 1개 강의 100% 이수 시 이수증 발급



2. 교육 이수 방법(붙임의 매뉴얼 참조)

  로그인 → LMS 바로가기 (왼쪽 메뉴 영역 중간) → 마이페이지 (Learning)

  → 2025년 교직원 대상 아동학대 예방교육→ 과목 홈 바로가기 → 주차학습

  → 강의 수강 (한국어, 영어 중 택1 가능)



3. 유의사항

  가. 전임교원 이수결과 교육업적평가 반영

  나. 정규직 직원 이수결과 직원인사고과 평가 반영

  다. 학습기간 중 미이수자에 대한 독려 문자 및 메일 발송 예정.



학부대학 교육운영팀
유관순 조교/ 02-000-1111

"""

In [ ]:
# ==========================================
# 1. 라이브러리 임포트
# ==========================================
import os  # 환경변수(API KEY) 설정용
import gradio as gr  # UI 구성
from google.colab import userdata  # Colab secrets 접근

from langchain_openai import ChatOpenAI  # OpenAI LLM
from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿
from langchain_core.output_parsers import StrOutputParser  # LLM 출력 문자열 변환

# ==========================================
# 2. API KEY 설정
# ==========================================
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # OpenAI 키 로드

# ==========================================
# 3. LLM 초기화
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 역할: 이메일 생성 및 분석을 수행하는 핵심 LLM

# ==========================================
# 4. LCEL 실행기
# ==========================================
def run_chain(template, inputs):
    """
    역할:
    - Prompt → LLM → OutputParser를 연결한 LCEL 실행기
    - 모든 Agent에서 공통으로 사용
    """

    prompt =


    chain =


    return


# ==========================================
# 5. 이메일 분리 Agent
# ==========================================
def split_email(full_email):
    """
    역할 (Agent):
    - 통합 이메일 입력을 Header / Body로 자동 분리
    - LLM 기반 구조 분석 Agent
    """

    template = """
다음 이메일을 분석해서 반드시 아래 형식으로 출력하세요.

HEADER: <헤더 내용>
BODY: <본문 내용>

이메일:
{email}
"""

    result = run_chain(template, {"email": full_email})

    header = ""  # 추출된 헤더 저장 변수
    body = ""    # 추출된 본문 저장 변수

    # LLM 출력 파싱
    for line in result.splitlines():


    return header, body


# ==========================================
# 6. 발신자/날짜 추출 Agent
# ==========================================
def parse_email_header(header):
    """
    역할 (Agent):
    - 이메일 헤더에서 발신자 이름과 수신 날짜 추출
    """

    template = """
헤더에서 아래 정보를 추출하세요.

출력 형식:
Sender Name: <이름>
Received Date: <날짜>

헤더:
{header}
"""

    out =


    sender =

    date =


    # 출력 파싱
    for line in out.splitlines():


    return sender, date


# ==========================================
# 7. 수신자 역할 추론 Agent
# ==========================================
def detect_role(header, body):
    """
    역할 (Agent):
    - 이메일 문맥 기반 수신자 역할 추론
    - 상사 / 동료 / 고객 / 공급업체 / 기타
    """

    template = """
이메일 헤더와 본문을 보고 수신자 역할을 판단하세요.

범주:
상사 / 동료 / 고객 / 공급업체 / 기타

출력:
Role: <역할>

헤더:
{header}

본문:
{body}
"""

    out =


    # Role 값만 추출
    return



# ==========================================
# 8. 이메일 요약 Agent
# ==========================================
def summarize_email(body):
    """
    역할 (Agent):
    - 이메일 본문 요약 생성
    """

    template = "다음 이메일을 간결하게 요약하세요:\n\n{body}"

    return



# ==========================================
# 9. 답변 생성 Agent
# ==========================================
def draft_reply(body, summary, sender, date, role):
    """
    역할 (Agent):
    - 모든 정보를 기반으로 이메일 답변 생성
    """

    template = """
이메일 내용: {body}
요약: {summary}
발신자: {sender}
수신일: {date}
수신자 역할: {role}

위 정보를 기반으로 전문적이고 간결한 답변을 작성하세요.
"""

    return run_chain(template, {


    })


# ==========================================
# 10. 이메일 태깅 Agent
# ==========================================
def tag_email(body, summary):
    """
    역할 (Agent):
    - 이메일 유형 분류
    - 긴급 / 일반 / 홍보 / 기타
    """

    template = """
이메일을 다음 중 하나로 분류하세요:
긴급 / 일반 / 홍보 / 기타

출력:
Tag: <태그>

본문:
{body}

요약:
{summary}
"""

    out =

    return out.split("Tag:")[-1].strip() if "Tag:" in out else "기타"


# ==========================================
# 11. 전체 Agent Pipeline
# ==========================================
def process_email(full_email):
    """
    역할:
    - 전체 이메일 처리 파이프라인
    - 단일 입력 → 자동 분리 → 분석 → 생성 → 분류
    """

    # 입력 검증
    if not full_email.strip():
        return ("", "", "", "", "", "")

    # 1단계: 이메일 구조 분리 Agent 실행
    header, body =

    # 2단계: 메타 정보 추출 Agent
    sender, date =

    # 3단계: 역할 추론 Agent
    role =

    # 4단계: 요약 Agent
    summary =


    # 5단계: 답변 생성 Agent
    reply =


    # 6단계: 태깅 Agent
    tag =


    return sender, date, role, summary, reply, tag


# ==========================================
# 12. Gradio UI
# ==========================================
with gr.Blocks() as demo:

    gr.Markdown("## 이메일 자동 Agent (LCEL 기반 구조)")

    email_input = gr.Textbox(
        label="통합 이메일 입력",
        lines=12,
        placeholder="헤더 + 본문 전체를 입력"
    )

    btn = gr.Button("실행")

    sender_out = gr.Textbox(label="발신자")
    date_out = gr.Textbox(label="수신일")
    role_out = gr.Textbox(label="수신자 역할")
    summary_out = gr.Textbox(label="요약")
    reply_out = gr.Textbox(label="답변")
    tag_out = gr.Textbox(label="태그")

    btn.click(
        fn=process_email,
        inputs=email_input,
        outputs=[sender_out, date_out, role_out, summary_out, reply_out, tag_out]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d47937c3ad39fa39a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
"""

RAG + Langgraph 하이브리드 구조

            ┌───── ─────┐
            │  PDF / Web Loader   │
            └────┬─ ────┘
                      ↓
            ┌───── ─────┐
            │   Vector DB (FAISS) │
            └────┬─── ──┘
                      ↓
                [RAG Retriever]
                      ↓
      ┌────────────────┐
      │        LangGraph Agent         │
      │                                │
      │  Parse → Retrieve → Generate │
      │            ↓                  │
      │         Review Agent           │
      │      (quality judge)           │
      │            ↓                  │
      │   approve / rewrite loop       │
      └────────────────┘

In [ ]:
!pip install -q langgraph langchain-openai langchain-community faiss-cpu pypdf beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.0 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 1. 라이브러리 임포트
# ==========================================
import os  # 환경 변수 관리
import requests  # (확장용: web scraping 가능)
from bs4 import BeautifulSoup  # (확장용: HTML parsing)

from typing import TypedDict  # LangGraph State 정의

# LangChain 핵심 구성 요소
from langchain_openai import ChatOpenAI  # LLM
from langchain_core.prompts import PromptTemplate  # Prompt template
from langchain_core.output_parsers import StrOutputParser  # 문자열 출력 파서
from langchain_core.documents import Document  # RAG 문서 구조

# Vector DB (RAG)
from langchain_community.vectorstores import FAISS  # FAISS 벡터 DB
from langchain_community.embeddings import OpenAIEmbeddings  # embedding 모델

# LangGraph
from langgraph.graph import StateGraph, END  # workflow engine


# ==========================================
# 2. API KEY 설정
# ==========================================
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# ==========================================
# 3. LLM 초기화
# ==========================================
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 경량 고성능 모델
    temperature=0         # 결정적 출력 (재현성 확보)
)


# ==========================================
# 4. RAG 데이터셋 (샘플 문서)
# ==========================================
# 실제 PDF/Web 대신 테스트용 정책 문서 역할
docs = [
    Document(page_content="이메일은 항상 정중하고 명확하게 작성해야 한다"),
    Document(page_content="회의 일정 변경 시 최소 24시간 전에 통보한다"),
    Document(page_content="고객 응대 이메일은 공손한 표현을 사용해야 한다")
]


# ==========================================
# 5. Vector DB 생성 (FAISS RAG)
# ==========================================
vectordb =

retriever =


# ==========================================
# 6. LangGraph State 정의
# ==========================================
class EmailState(TypedDict):
    email_text: str   # 사용자 입력
    context: str      # RAG 검색 결과
    draft: str        # 생성된 이메일 초안
    score: int        # 품질 평가 점수
    approved: bool    # 승인 여부


# ==========================================
# 7. Agent 1 - Retrieve Node (RAG 검색 에이전트)
# ==========================================
def retrieve_node(state: EmailState):
    """
    역할:
    - 사용자 입력을 기반으로 FAISS에서 관련 문서 검색
    - 이메일 작성에 필요한 context 생성
    """

    docs =

    context = "\n".join([d.page_content for d in docs])  # 텍스트 결합

    return {  }


# ==========================================
# 8. Agent 2 - Generate Node (이메일 생성 에이전트)
# ==========================================
def generate_node(state: EmailState):
    """
    역할:
    - RAG context + 사용자 요청 기반 이메일 생성
    """

    prompt = PromptTemplate.from_template(
        """
        아래 참고자료를 기반으로 이메일을 작성하세요.

        [참고자료]
        {context}

        [요청]
        {email_text}
        """
    )

    chain = prompt | llm | StrOutputParser()  # LCEL pipeline

    draft =


    return {  }


# ==========================================
# 9. Agent 3 - Review Node (LLM Judge / 평가자)
# ==========================================
def review_node(state: EmailState):
    """
    역할:
    - 생성된 이메일 품질 평가
    - 0~10 점수 기반 승인 여부 판단
    """

    prompt = PromptTemplate.from_template(
        """
        이메일 품질을 0~10으로 평가하세요.

        기준:
        - 정확성
        - 자연스러움
        - 참고자료 반영 여부

        이메일:
        {draft}

        숫자만 출력하세요.
        """
    )

    chain = prompt | llm | StrOutputParser()

    score_raw =


    # 안전한 정수 변환
    try:
        score = int(score_raw.strip())
    except:
        score = 0

    return {
        "score": score,
        "approved": score >= 7  # 7점 이상이면 승인
    }


# ==========================================
# 10. Agent 4 - Rewrite Node (개선 에이전트)
# ==========================================
def rewrite_node(state: EmailState):
    """
    역할:
    - 평가 실패 시 이메일 개선
    - 반복 loop 개선 구조
    """

    prompt = PromptTemplate.from_template(
        """
        아래 이메일을 더 개선하세요.

        - 더 전문적으로
        - 더 명확하게
        - 참고자료 반영 강화

        이메일:
        {draft}
        """
    )

    chain = prompt | llm | StrOutputParser()

    new_draft =

    return {  }


# ==========================================
# 11. LangGraph Workflow 구성
# ==========================================
builder = StateGraph(EmailState)  # 상태 기반 그래프 생성


# 노드 등록
builder.add_node("retrieve", retrieve_node)   # RAG 검색
builder.add_node("generate", generate_node)   # 이메일 생성
builder.add_node("review", review_node)       # 평가
builder.add_node("rewrite", rewrite_node)     # 개선


# 시작 노드
builder.set_entry_point("retrieve")


# ==========================================
# 12. Edge 연결 (Flow 정의)
# ==========================================
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "review")


# ==========================================
# 13. Routing (Decision Logic)
# ==========================================
def route(state):
    """
    역할:
    - review 결과 기반 workflow 제어
    - 승인 or 개선 루프 결정
    """
    return "END" if state["approved"] else "rewrite"


builder.add_conditional_edges(
    "review",
    route,
    {
        "END": END,          # 종료
        "rewrite": "rewrite" # 개선 루프
    }
)

# rewrite → review loop (iterative improvement)
builder.add_edge("rewrite", "review")


# ==========================================
# 14. Compile (실행 가능한 Agent 생성)
# ==========================================
app = builder.compile()


# ==========================================
# 15. 실행 테스트
# ==========================================
result = app.invoke({
    "email_text": "회의 일정 변경 요청 이메일 작성"
})


# ==========================================
# 16. 결과 출력
# ==========================================
print("\n================ 결과 =================")
print(result["draft"])      # 최종 이메일
print("Score:", result["score"])  # 평가 점수
print("Approved:", result["approved"])  # 승인 여부


================ 결과 =================
제목: 회의 일정 변경 요청

안녕하세요, [고객님 성함]님.

저희 [회사명]의 [당신의 이름]입니다. 

먼저, 귀하와의 회의에 대한 소중한 시간을 할애해 주셔서 감사드립니다. 다름이 아니라, 예정되어 있던 회의 일정에 대해 변경 요청을 드리고자 합니다. 

현재 회의는 [기존 일정]에 예정되어 있으나, 부득이한 사정으로 인해 [변경 사유]로 인해 일정을 조정해야 할 필요가 생겼습니다. 이에 따라, 회의를 [새로운 일정]으로 변경해 주실 수 있는지 여쭙고자 합니다. 

변경된 일정이 가능하시다면, 저희에게도 큰 도움이 될 것입니다. 만약 새로운 일정이 어려우시다면, 귀하의 편하신 시간에 맞춰 조정하도록 하겠습니다. 

회의 일정 변경 요청은 최소 24시간 전에 통보드리는 것이 원칙임을 잘 알고 있습니다. 이 점 양해 부탁드리며, 최대한 빠른 시일 내에 회의 일정을 확정할 수 있도록 노력하겠습니다.

귀하의 소중한 의견을 기다리겠습니다. 감사합니다.

좋은 하루 되세요.

[당신의 이름]  
[당신의 직책]  
[회사명]  
[연락처]  
[이메일 주소]  
Score: 8
Approved: True
